In [2]:
import os
for dp, dirs, files in os.walk("/kaggle/input"):
    if "My Dataset" in dp:      # skip the big image dataset
        continue
    for f in files:
        print(os.path.join(dp, f))

/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default/1/config.json
/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default/1/metadata.json
/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default/1/model.weights.h5


In [3]:
import os
for dp, dirs, files in os.walk("/kaggle/input"):
    if "My Dataset" in dp:          # skip the big image dataset
        continue
    print(dp)
    for f in files:
        print("    ", f)

/kaggle/input
/kaggle/input/models
/kaggle/input/models/harshkumar970
/kaggle/input/models/harshkumar970/my-model-v6
/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2
/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default
/kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default/1
     config.json
     metadata.json
     model.weights.h5


In [4]:
import os, shutil, zipfile, glob
import tensorflow as tf

# find the folder that contains config.json (that is the unpacked model)
src = None
for dp, dirs, files in os.walk("/kaggle/input"):
    if "My Dataset" in dp:
        continue
    if "config.json" in files and "model.weights.h5" in files:
        if "best_model" in dp:
            src = dp; break
        src = src or dp
print("Unpacked model folder:", src)

MODEL_PATH = "/kaggle/working/best_model.keras"
with zipfile.ZipFile(MODEL_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(src):
        z.write(os.path.join(src, fn), arcname=fn)     # files at the top level of the archive

model = tf.keras.models.load_model(MODEL_PATH)
IMG = model.input_shape[1]
NUM = model.output_shape[-1]
print("Loaded OK | Input size:", IMG, "| Classes:", NUM)

Unpacked model folder: /kaggle/input/models/harshkumar970/my-model-v6/tensorflow2/default/1
Loaded OK | Input size: 224 | Classes: 5


In [5]:
json_files = glob.glob("/kaggle/input/**/classes.json", recursive=True)
print(json_files)

if json_files:
    classes = json.load(open(json_files[0]))
else:
    # no classes.json attached: type the names exactly as your training notebook printed them
    classes = ["Animal", "Disaster", "Human", "Plants", "Vehical"]   # <- edit to match

assert len(classes) == NUM, f"{len(classes)} names but the model has {NUM} outputs"
print(classes)

[]
['Animal', 'Disaster', 'Human', 'Plants', 'Vehical']


In [6]:
!pip install pyngrok
from pyngrok import ngrok
!pip install -q flask flask-cors pyngrok pillow
ngrok.set_auth_token("3LlxzzxVUZbVZlg8B5eysBhu7G9_6nGgnAYpdkuGMcTCJrUDt")

import io, os, json, time, socket, threading, requests
import numpy as np
from PIL import Image
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import tensorflow as tf

# ---------- 1) Model (use the one in memory, else load from file) ----------
try:
    model
except NameError:
    MODEL_PATH = "/kaggle/working/best_model.keras"   # change if your file is named differently
    model = tf.keras.models.load_model(MODEL_PATH)

# image size is read from the model itself, so it always matches training
img_height, img_width = model.input_shape[1], model.input_shape[2]
NUM_CLASSES = model.output_shape[-1]

# ---------- 2) Class names (must be in the SAME order as training) ----------
class_names = None
for getter in (
    lambda: list(train_ds.class_names),
    lambda: list(classes),
    lambda: json.load(open("/kaggle/working/classes.json")),
    lambda: json.load(open("/kaggle/working/class_names.json")),
):
    try:
        class_names = getter()
        break
    except Exception:
        pass

if class_names is None:
    raise RuntimeError("Class names not found. Set: class_names = [...] in the same order as training.")

assert len(class_names) == NUM_CLASSES, f"{len(class_names)} names but model outputs {NUM_CLASSES}"
print(f"Model input: {img_height}x{img_width}, {NUM_CLASSES} classes")

# ---------- 3) ngrok token (use Kaggle Secrets, never paste the token here) ----------
try:
    from kaggle_secrets import UserSecretsClient
    ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
except Exception as e:
    print("Secret NGROK_TOKEN not found, using existing ngrok config:", e)

# ---------- 4) Flask app ----------
app = Flask(__name__)
CORS(app)

def preprocess(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((img_width, img_height))
    arr = np.array(img, dtype="float32")        # keep 0-255 (model has preprocessing inside)
    return np.expand_dims(arr, axis=0)

@app.route("/")
def home():
    return "Classifier API is running"

@app.route("/health")
@app.route("/api/health")
@app.route("/ping")
def health():
    return jsonify({"status": "ok", "ok": True})

@app.route("/predict", methods=["POST", "OPTIONS"])
@app.route("/api/predict", methods=["POST", "OPTIONS"])
def predict():
    if request.method == "OPTIONS":
        return ("", 204)

    f = request.files.get("file") or request.files.get("image")
    if f is None:
        return jsonify({"error": "No file uploaded"}), 400

    try:
        probs = model.predict(preprocess(f.read()), verbose=0)[0]
        top = np.argsort(probs)[::-1][:3]
        top3 = [{
            "class": class_names[int(i)],
            "label": class_names[int(i)],
            "name": class_names[int(i)],
            "confidence": float(probs[i]),
            "probability": float(probs[i]),
            "score": float(probs[i]),
        } for i in top]

        best = top3[0]
        return jsonify({
            "prediction": best["class"],
            "predicted_class": best["class"],
            "class": best["class"],
            "label": best["class"],
            "confidence": best["confidence"],
            "probability": best["confidence"],
            "top3": top3,
            "top_3": top3,
            "top_predictions": top3,
            "predictions": top3,
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.after_request
def add_headers(resp):
    resp.headers["Access-Control-Allow-Origin"] = "*"
    resp.headers["Access-Control-Allow-Headers"] = "*"
    resp.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
    return resp

# ---------- 5) Start server on a free port ----------
def free_port():
    s = socket.socket()
    s.bind(("", 0))
    p = s.getsockname()[1]
    s.close()
    return p

PORT = free_port()
threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=PORT, use_reloader=False),
    daemon=True
).start()
time.sleep(3)
print("Local test:", requests.get(f"http://127.0.0.1:{PORT}/").text)

# ---------- 6) Public URL ----------
ngrok.kill()
public_url = ngrok.connect(PORT).public_url
print("\n✅ Your API URL:", public_url)
print("Predict endpoint:", public_url + "/predict")

Model input: 224x224, 5 classes                                                                     
Secret NGROK_TOKEN not found, using existing ngrok config: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 143354168 and label NGROK_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:58227
 * Running on http://172.19.2.2:58227
Press CTRL+C to quit
127.0.0.1 - - [08/Nov/2026 18:42:24] "GET / HTTP/1.1" 200 -


Local test: Classifier API is running

✅ Your API URL: https://fabric-oval-comfy.ngrok-free.dev
Predict endpoint: https://fabric-oval-comfy.ngrok-free.dev/predict


127.0.0.1 - - [08/Nov/2026 18:43:21] "OPTIONS /health HTTP/1.1" 200 -
127.0.0.1 - - [08/Nov/2026 18:43:22] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [08/Nov/2026 18:43:36] "OPTIONS /predict HTTP/1.1" 204 -
127.0.0.1 - - [08/Nov/2026 18:43:39] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [08/Nov/2026 18:44:06] "OPTIONS /predict HTTP/1.1" 204 -
127.0.0.1 - - [08/Nov/2026 18:44:07] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [08/Nov/2026 18:44:22] "OPTIONS /predict HTTP/1.1" 204 -
127.0.0.1 - - [08/Nov/2026 18:44:23] "POST /predict HTTP/1.1" 200 -
